In [1]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

In [2]:
import optuna
import pickle
from functools import partial
from pathlib import Path

from simulator.simulation.modules import Campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check

/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

In [4]:
from simulator.model.rlb_dp_bidder import RLBDPBidder

In [5]:
auction_mode = "FPA"  # or "VCG"
best_params_subfolder = f"{auction_mode.lower()}_rlb_ta_n10_rndm_42"
best_models_subfolder = f"{auction_mode.lower()}_rlb_ta_n10_rndm_42"

# metric to optimize: CPC_REL / RMSE / SCR
metric = "SCR"
n_trials = 1

In [6]:
data_config = {
    "train": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_train_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_train_final.csv",
    },
    "test": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_test_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_test_final.csv",
    },
}

data_config

{'train': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_train_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_train_final.csv'},
 'test': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_test_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_test_final.csv'}}

In [7]:
stats_path = data_config['train']['stats_path']
campaigns_path = data_config['train']['campaigns_path']

In [8]:
stats_df = pd.read_csv(stats_path)

In [9]:
import csv
import os
import uuid
from datetime import datetime


ARTIFACT_DIR = "tmp_models"
TRIALS_LOG_PATH = os.path.join(ARTIFACT_DIR, "optuna_trials.csv")


def objective_rlb_dp(trial, metric='RMSE_T', auction_mode='FPA'):

    max_bid = trial.suggest_float('max_bid', 10, 500, log=True)
    gamma = trial.suggest_float('gamma', 0.80, 1.00)
    N_bound = trial.suggest_int('N_bound', 6, 72)
    B_bound = trial.suggest_int('B_bound', 1e3, 2e4, log=True)

    custom_params = {
        "max_bid": max_bid,
        "lower_clip": 5,
        "upper_clip": 5,
        "gamma": gamma,
        "model_path": None,
        "N_bound": N_bound,
        "B_bound": B_bound,
        "use_traffic_share_state": True,
        "traffic_share_mode": "remaining_ratio",
    }

    bidder = RLBDPBidder(custom_params)

    bidder.fit(stats_df, campaign_path=campaigns_path)

    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    tmp_model_path = os.path.join(ARTIFACT_DIR, f"rlb_dp_trial{trial.number}_{uuid.uuid4().hex}.pkl")
    bidder.save_model(tmp_model_path)

    # прогоняем через тот же пайплайн проверки
    res = autobidder_check(
        bidder=RLBDPBidder,
        params={
            "input_campaigns": campaigns_path,
            "input_stats": stats_path,
            "max_bid": max_bid,
            "gamma": gamma,
            "model_path": tmp_model_path,
            "N_bound": N_bound,
            "B_bound": B_bound,
            "use_traffic_share_state": True,
            "traffic_share_mode": "remaining_ratio",
        },
        auction_mode=auction_mode,
    )

    score_map = {
        'RMSE_T': float(res['score'][1]),
        'CPC_REL': float(res['score'][0]),
        'SCR': float(res['score'][2]),
    }
    objective_value = score_map[metric]

    # Сохраняем метрики в trial attrs, чтобы callback мог централизованно логировать.
    trial.set_user_attr('tmp_model_path', tmp_model_path)
    trial.set_user_attr('score_cpc_rel', float(res['score'][0]))
    trial.set_user_attr('score_rmse_t', float(res['score'][1]))
    trial.set_user_attr('score_scr', float(res['score'][2]))

    return objective_value


def _optuna_progress_callback(study, trial):
    os.makedirs(ARTIFACT_DIR, exist_ok=True)

    row = {
        'ts': datetime.utcnow().isoformat(),
        'trial': int(trial.number),
        'state': str(trial.state),
        'objective_value': float(trial.value) if trial.value is not None else None,
        'best_value': float(study.best_value) if study.best_trial is not None else None,
        'score_cpc_rel': trial.user_attrs.get('score_cpc_rel'),
        'score_rmse_t': trial.user_attrs.get('score_rmse_t'),
        'score_scr': trial.user_attrs.get('score_scr'),
        'tmp_model_path': trial.user_attrs.get('tmp_model_path'),
    }

    file_exists = os.path.exists(TRIALS_LOG_PATH)
    with open(TRIALS_LOG_PATH, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

    print(
        f"[trial {trial.number}] value={row['objective_value']} | "
        f"best={row['best_value']} | model={row['tmp_model_path']}"
    )


def opt_search_rlb_dp(n_trials, metric='RMSE_T', auction_mode='FPA', n_jobs=6):
    optuna.logging.set_verbosity(optuna.logging.INFO)

    study = optuna.create_study(
        direction='maximize' if metric == 'SCR' else 'minimize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )

    study.optimize(
        partial(objective_rlb_dp, metric=metric, auction_mode=auction_mode),
        n_trials=n_trials,
        n_jobs=n_jobs,
        callbacks=[_optuna_progress_callback],
    )

    print('Best trial:')
    trial = study.best_trial
    print(f'  Value: {trial.value}')
    print('  Params: ')

    dict_path = f'best_params/rlb_dp_{metric.lower()}_{auction_mode}.pkl'
    params_dict = {}
    for key, value in trial.params.items():
        print(f'    {key}: {value}')
        params_dict[key] = value

    with open(dict_path, 'wb') as f:
        pickle.dump(params_dict, f)

    return study


def train_best_rlb_dp(best_params_path, model_path='rlb_dp_model_tuned.pkl'):
    """Обучить и сохранить модель с лучшими параметрами (после optuna)."""
    with open(best_params_path, 'rb') as f:
        best_params = pickle.load(f)

    custom_params = {
        "max_bid": best_params["max_bid"],
        "gamma": best_params["gamma"],
        "model_path": None,
        "N_bound": best_params["N_bound"],
        "B_bound": best_params["B_bound"],
        "use_traffic_share_state": True,
        "traffic_share_mode": "remaining_ratio",
    }

    bidder = RLBDPBidder(custom_params)
    bidder.fit(stats_df, campaign_path=campaigns_path)
    bidder.save_model(model_path)
    return bidder


In [10]:
study_rlb = opt_search_rlb_dp(n_trials, metric, auction_mode)

[I 2026-04-29 00:51:33,364] A new study created in memory with name: no-name-0e7eb05e-587b-43ab-b856-f0e32604fb9c
HoursTraffic: 100%|██████████| 31/31 [00:00<00:00, 43.63it/s]
[I 2026-04-29 01:10:17,549] Trial 0 finished with value: 9774.002743361965 and parameters: {'max_bid': 143.80849316565855, 'gamma': 0.8781365638465268, 'N_bound': 31, 'B_bound': 2127}. Best is trial 0 with value: 9774.002743361965.


[trial 0] value=9774.002743361965 | best=9774.002743361965 | model=tmp_models/rlb_dp_trial0_caadbdaf6162465d99a2700fd4a577a0.pkl
Best trial:
  Value: 9774.002743361965
  Params: 
    max_bid: 143.80849316565855
    gamma: 0.8781365638465268
    N_bound: 31
    B_bound: 2127


In [11]:
best_params_path = f'best_params/{best_params_subfolder}/{metric.lower()}.pkl'
# best_params_path='best_params/rlb_dp_scr_FPA.pkl'
best_model_path = f'best_models/{best_models_subfolder}/{metric.lower()}.pkl'

rlb_bidder_best = train_best_rlb_dp(best_params_path, best_model_path)

FileNotFoundError: [Errno 2] No such file or directory: 'best_params/fpa_rlb_ta_n10_rndm_42/scr.pkl'

In [ ]:
best_params_path

'best_params/rlb_dp_scr_FPA.pkl'

In [ ]:
best_params_rlb = pd.read_pickle(best_params_path)
best_model_rlb_path = best_model_path

In [ ]:
campaigns_path_test = data_config['test']['campaigns_path']
stats_path_test = data_config['test']['stats_path']

In [ ]:
res = autobidder_check(
    bidder=RLBDPBidder,
    params = {
        "input_campaigns": campaigns_path_test,
        "input_stats": stats_path_test,
        "model_path": best_model_path,
        **best_params_rlb
    },
    auction_mode=auction_mode,
)

In [ ]:
print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")

CPC_REL: 1044.7359370385877, rmse: 1.2820161404703379, SCR: 15081.728715934123
